# 02 - Calidad de datos

En este notebook se evaluará la calidad de los datos integrados del proyecto EdTech Learning Analytics.

Se analizarán principalmente:

- completitud,
- unicidad,
- consistencia,
- validez,
- integridad,
- distribución de valores,
- y posibles valores atípicos.

El objetivo es identificar problemas que puedan afectar los análisis posteriores y definir reglas de tratamiento sin modificar innecesariamente la información original.

## 1. Importación de librerías

Se cargan las librerías necesarias para análisis, validación y visualización de calidad de datos.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Entorno listo correctamente")

pandas: 3.1.0
numpy: 2.5.4
Entorno listo correctamente


## 2. Configuración de rutas

Se definen las rutas principales del proyecto para acceder a los datasets procesados generados en el notebook anterior.

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_PROCESSED = DATA_DIR / "procesados"

ruta_integrado = DATA_PROCESSED / "edtech_integrado.csv"
ruta_student_analysis = DATA_PROCESSED / "edtech_student_analysis.csv"

print("Dataset integrado:", ruta_integrado)
print("Dataset analítico:", ruta_student_analysis)

Dataset integrado: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_integrado.csv
Dataset analítico: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_student_analysis.csv


In [3]:
archivos = {
    "edtech_integrado.csv": ruta_integrado,
    "edtech_student_analysis.csv": ruta_student_analysis
}

for nombre, ruta in archivos.items():
    estado = "OK" if ruta.exists() else "NO ENCONTRADO"
    print(f"{nombre:35} -> {estado}")

edtech_integrado.csv                -> OK
edtech_student_analysis.csv         -> OK


## 3. Carga de datasets procesados

Se cargan los datasets generados en la etapa anterior para evaluar su calidad sin repetir el proceso de integración.

In [4]:
df_integrado = pd.read_csv(ruta_integrado)
df_student_analysis = pd.read_csv(ruta_student_analysis)

print("Datasets cargados correctamente")

print("Integrado:", df_integrado.shape)
print("Student analysis:", df_student_analysis.shape)

Datasets cargados correctamente
Integrado: (200000, 31)
Student analysis: (8000, 20)


In [5]:
display(df_integrado.head())
display(df_student_analysis.head())

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_title,subject_area,difficulty_level,duration_weeks,instructor_rating,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Economics 384,Computer Science,Advanced,4,4.4,Video,1,1,0,1
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Music 579,English,Beginner,4,4.7,Mixed,1,1,1,1
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Language Learning 231,History,Beginner,7,3.9,Text-based,1,1,0,0
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Biology 496,Art,Intermediate,5,3.8,Mixed,0,1,0,1
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Psychology 251,Computer Science,Advanced,4,4.7,Video,1,1,1,1


,student_id,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,motivation_level,total_interactions,total_duration_minutes,average_duration_seconds,average_completion_rate,average_score,evaluated_activities,average_attempts,help_sought_count,peer_interaction_count,below_average_score
0,student_00000,46,Undergraduate,Social Sciences,Reading/Writing,Desktop,UTC-5,1,0,Medium,22,269.75,735.68,0.9014,0.8375,8,1.32,3,2,False
1,student_00001,16,High School,Social Sciences,Auditory,Mobile,UTC+8,6,1,Low,23,321.78,839.43,0.8635,0.7936,11,1.65,1,0,True
2,student_00002,27,Graduate,STEM,Reading/Writing,Mobile,UTC-5,7,0,High,26,267.77,617.92,0.8319,0.8614,7,1.23,4,0,False
3,student_00003,57,Undergraduate,Arts,Visual,Desktop,UTC-8,3,0,High,20,207.25,621.75,0.8035,0.7900,4,1.20,2,2,True
4,student_00004,39,High School,STEM,Auditory,Tablet,UTC,8,1,High,23,375.15,978.65,0.8643,0.8014,7,1.17,4,2,False


## 4. Diagnóstico general de calidad

Se realiza una evaluación inicial considerando dimensiones de calidad como completitud, unicidad, consistencia y validez.

In [6]:
def resumen_calidad(df):
    return pd.DataFrame({
        "tipo_dato": df.dtypes.astype(str),
        "valores_nulos": df.isnull().sum(),
        "porcentaje_nulos": (df.isnull().mean() * 100).round(2),
        "valores_unicos": df.nunique(),
        "duplicados_columna": [
            df[col].duplicated().sum()
            for col in df.columns
        ]
    })

resumen_integrado = resumen_calidad(df_integrado)
resumen_student = resumen_calidad(df_student_analysis)

display(resumen_integrado)

,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos,duplicados_columna
interaction_id,str,0,0.00,200000,0
student_id,str,0,0.00,8000,192000
course_id,str,0,0.00,150,199850
timestamp,str,0,0.00,196699,3301
activity_type,str,0,0.00,6,199994
duration_seconds,int64,0,0.00,4185,195815
completion_rate,float64,0,0.00,101,199899
score,float64,133419,66.71,41,199958
attempts,int64,0,0.00,3,199997
device_used,str,0,0.00,3,199997


In [7]:
display(resumen_student)

,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos,duplicados_columna
student_id,str,0,0.00,8000,0
age,int64,0,0.00,49,7951
education_level,str,0,0.00,4,7996
study_field,str,0,0.00,5,7995
learning_style,str,0,0.00,4,7996
device_preference,str,0,0.00,3,7997
timezone,str,0,0.00,5,7995
previous_online_courses,int64,0,0.00,16,7984
has_learning_disability,int64,0,0.00,2,7998
motivation_level,str,0,0.00,3,7997


## 5. Completitud de los datos

En esta sección se analiza la presencia de valores nulos en los datasets procesados.

El objetivo es distinguir entre valores faltantes que representan problemas de calidad y valores nulos que son válidos por la naturaleza de la información.

In [8]:
def analizar_completitud(df):
    resumen = pd.DataFrame({
        "valores_nulos": df.isnull().sum(),
        "porcentaje_nulos": (df.isnull().mean() * 100).round(2),
        "valores_no_nulos": df.notnull().sum()
    })

    resumen["completitud_pct"] = (
        100 - resumen["porcentaje_nulos"]
    ).round(2)

    return resumen.sort_values(
        by="porcentaje_nulos",
        ascending=False
    )

In [9]:
completitud_integrado = analizar_completitud(df_integrado)

display(completitud_integrado)

,valores_nulos,porcentaje_nulos,valores_no_nulos,completitud_pct
score,133419,66.71,66581,33.29
student_id,0,0.00,200000,100.00
interaction_id,0,0.00,200000,100.00
timestamp,0,0.00,200000,100.00
activity_type,0,0.00,200000,100.00
duration_seconds,0,0.00,200000,100.00
course_id,0,0.00,200000,100.00
completion_rate,0,0.00,200000,100.00
attempts,0,0.00,200000,100.00
device_used,0,0.00,200000,100.00


In [10]:
completitud_student = analizar_completitud(df_student_analysis)

display(completitud_student)

,valores_nulos,porcentaje_nulos,valores_no_nulos,completitud_pct
average_score,2,0.02,7998,99.98
student_id,0,0.00,8000,100.00
education_level,0,0.00,8000,100.00
age,0,0.00,8000,100.00
study_field,0,0.00,8000,100.00
learning_style,0,0.00,8000,100.00
timezone,0,0.00,8000,100.00
device_preference,0,0.00,8000,100.00
has_learning_disability,0,0.00,8000,100.00
motivation_level,0,0.00,8000,100.00


### 5.1 Identificación de columnas con valores nulos

Se identifican únicamente las variables que contienen valores faltantes para facilitar su análisis.

In [11]:
columnas_nulas_integrado = completitud_integrado[
    completitud_integrado["valores_nulos"] > 0
]

columnas_nulas_student = completitud_student[
    completitud_student["valores_nulos"] > 0
]

print("Columnas con nulos - Dataset integrado")
display(columnas_nulas_integrado)

print("\nColumnas con nulos - Dataset analítico por estudiante")
display(columnas_nulas_student)

Columnas con nulos - Dataset integrado


,valores_nulos,porcentaje_nulos,valores_no_nulos,completitud_pct
score,133419,66.71,66581,33.29



Columnas con nulos - Dataset analítico por estudiante


,valores_nulos,porcentaje_nulos,valores_no_nulos,completitud_pct
average_score,2,0.02,7998,99.98


### 5.2 Evaluación específica de `score`

La variable `score` contiene valores nulos debido a que no todas las actividades generan una calificación.

Por esta razón, se valida nuevamente si los valores faltantes están asociados únicamente a actividades no evaluables.

In [12]:
score_completitud = (
    df_integrado
    .groupby("activity_type")
    .agg(
        registros=("interaction_id", "count"),
        score_no_nulo=("score", "count"),
        score_nulo=("score", lambda x: x.isnull().sum())
    )
)

score_completitud["porcentaje_nulo"] = (
    score_completitud["score_nulo"]
    / score_completitud["registros"]
    * 100
).round(2)

display(score_completitud)

,registros,score_no_nulo,score_nulo,porcentaje_nulo
activity_type,,,,
Assignment Submit,33170,33170,0,0.0
Discussion Post,33328,0,33328,100.0
Note Taking,33754,0,33754,100.0
Quiz Attempt,33411,33411,0,0.0
Resource Download,33383,0,33383,100.0
Video Watch,32954,0,32954,100.0


In [13]:
actividades_con_score = (
    df_integrado
    .loc[df_integrado["score"].notnull(), "activity_type"]
    .unique()
)

actividades_sin_score = (
    df_integrado
    .loc[df_integrado["score"].isnull(), "activity_type"]
    .unique()
)

print("Actividades con score:")
print(actividades_con_score)

print("\nActividades sin score:")
print(actividades_sin_score)

Actividades con score:
<StringArray>
['Assignment Submit', 'Quiz Attempt']
Length: 2, dtype: str

Actividades sin score:
<StringArray>
['Note Taking', 'Resource Download', 'Video Watch', 'Discussion Post']
Length: 4, dtype: str


### Interpretación de completitud

Los valores nulos deben analizarse de acuerdo con el contexto de cada variable.

En el caso de `score`, los valores faltantes no representan necesariamente errores de calidad, ya que determinadas actividades de la plataforma no generan una calificación.

Por esta razón, no se realizará una imputación automática de estos valores. Para los análisis de rendimiento se utilizarán únicamente las actividades evaluables.

### 5.3 Completitud general del dataset

In [14]:
total_celdas_integrado = (
    df_integrado.shape[0] * df_integrado.shape[1]
)

total_nulos_integrado = (
    df_integrado.isnull().sum().sum()
)

completitud_general_integrado = (
    1 - total_nulos_integrado / total_celdas_integrado
) * 100


total_celdas_student = (
    df_student_analysis.shape[0]
    * df_student_analysis.shape[1]
)

total_nulos_student = (
    df_student_analysis.isnull().sum().sum()
)

completitud_general_student = (
    1 - total_nulos_student / total_celdas_student
) * 100


print(
    f"Completitud general - Dataset integrado: "
    f"{completitud_general_integrado:.2f}%"
)

print(
    f"Completitud general - Dataset por estudiante: "
    f"{completitud_general_student:.2f}%"
)

Completitud general - Dataset integrado: 97.85%
Completitud general - Dataset por estudiante: 100.00%


## 6. Unicidad y duplicados

En esta sección se analiza la presencia de registros duplicados y se valida la unicidad de los identificadores principales.

El objetivo es detectar repeticiones que puedan afectar los resultados de los análisis y confirmar que las llaves principales mantengan una relación consistente con cada registro.

### 6.1 Registros completamente duplicados

Se verifica si existen filas idénticas dentro de los datasets procesados.

In [15]:
duplicados_integrado = df_integrado.duplicated().sum()
duplicados_student = df_student_analysis.duplicated().sum()

print("Duplicados completos")
print(f"Dataset integrado:              {duplicados_integrado}")
print(f"Dataset analítico por estudiante: {duplicados_student}")

Duplicados completos
Dataset integrado:              0
Dataset analítico por estudiante: 0


### 6.2 Validación de identificadores únicos

Se verifica que los identificadores principales mantengan la unicidad esperada según el nivel de granularidad de cada dataset.

In [16]:
print("=== Dataset integrado ===")
print(f"Registros totales:        {len(df_integrado)}")
print(f"interaction_id únicos:    {df_integrado['interaction_id'].nunique()}")
print(f"student_id únicos:        {df_integrado['student_id'].nunique()}")
print(f"course_id únicos:         {df_integrado['course_id'].nunique()}")

print("\n=== Dataset analítico por estudiante ===")
print(f"Registros totales:        {len(df_student_analysis)}")
print(f"student_id únicos:        {df_student_analysis['student_id'].nunique()}")

=== Dataset integrado ===
Registros totales:        200000
interaction_id únicos:    200000
student_id únicos:        8000
course_id únicos:         150

=== Dataset analítico por estudiante ===
Registros totales:        8000
student_id únicos:        8000


### 6.3 Duplicados por llave principal

Se valida explícitamente si existen identificadores principales repetidos.

In [17]:
duplicados_interaction_id = (
    df_integrado["interaction_id"]
    .duplicated()
    .sum()
)

duplicados_student_id = (
    df_student_analysis["student_id"]
    .duplicated()
    .sum()
)

print(
    "interaction_id duplicados en dataset integrado:",
    duplicados_interaction_id
)

print(
    "student_id duplicados en dataset por estudiante:",
    duplicados_student_id
)

interaction_id duplicados en dataset integrado: 0
student_id duplicados en dataset por estudiante: 0


### 6.4 Posibles interacciones repetidas

Además de revisar el identificador principal, se evalúan posibles repeticiones lógicas considerando estudiante, curso, fecha y tipo de actividad.

Esto permite detectar registros que podrían representar la misma interacción aunque tengan identificadores diferentes.

In [18]:
columnas_interaccion_logica = [
    "student_id",
    "course_id",
    "timestamp",
    "activity_type"
]

duplicados_logicos = df_integrado.duplicated(
    subset=columnas_interaccion_logica,
    keep=False
)

cantidad_duplicados_logicos = duplicados_logicos.sum()

print(
    "Registros involucrados en posibles duplicados lógicos:",
    cantidad_duplicados_logicos
)

Registros involucrados en posibles duplicados lógicos: 0


In [19]:
if cantidad_duplicados_logicos > 0:
    display(
        df_integrado.loc[
            duplicados_logicos,
            [
                "interaction_id",
                "student_id",
                "course_id",
                "timestamp",
                "activity_type",
                "score",
                "duration_seconds"
            ]
        ]
        .sort_values(
            by=[
                "student_id",
                "course_id",
                "timestamp",
                "activity_type"
            ]
        )
        .head(20)
    )
else:
    print("No se detectaron posibles duplicados lógicos.")

No se detectaron posibles duplicados lógicos.


### 6.5 Frecuencia de interacciones por estudiante

Se revisa la cantidad de interacciones asociadas a cada estudiante para identificar comportamientos inusuales o concentraciones excesivas de registros.

In [20]:
interacciones_por_estudiante = (
    df_integrado
    .groupby("student_id")
    .size()
)

display(
    interacciones_por_estudiante
    .describe()
)

count    8000.000000
mean       25.000000
std         4.948263
min        11.000000
25%        22.000000
50%        25.000000
75%        28.000000
max        44.000000
dtype: float64

In [21]:
print("Mínimo de interacciones por estudiante:")
print(interacciones_por_estudiante.min())

print("\nMáximo de interacciones por estudiante:")
print(interacciones_por_estudiante.max())

print("\nPromedio:")
print(interacciones_por_estudiante.mean())

Mínimo de interacciones por estudiante:
11

Máximo de interacciones por estudiante:
44

Promedio:
25.0


### Resultado del análisis de unicidad

Se validó la unicidad de las llaves principales de acuerdo con la granularidad de cada dataset.

El dataset integrado utiliza `interaction_id` como identificador único de cada interacción, mientras que el dataset analítico utiliza `student_id` como identificador único por estudiante.

También se revisaron posibles duplicados completos y duplicados lógicos. Cualquier coincidencia lógica encontrada debe analizarse antes de eliminar registros, ya que podría representar múltiples eventos válidos y no necesariamente errores de calidad.

## 7. Validez y consistencia de valores

En esta sección se verifica que los datos se encuentren dentro de rangos válidos y que las variables categóricas mantengan valores consistentes.

El objetivo es detectar posibles errores de captura, valores fuera de dominio o inconsistencias de formato que puedan afectar los análisis posteriores.

### 7.1 Revisión de variables numéricas

Se analizan las principales variables numéricas para conocer sus valores mínimos, máximos y distribución general.

In [22]:
variables_numericas = [
    "age",
    "duration_seconds",
    "completion_rate",
    "score",
    "attempts",
    "previous_online_courses",
    "instructor_rating"
]

for columna in variables_numericas:
    if columna in df_integrado.columns:
        print(f"\n=== {columna} ===")
        display(df_integrado[columna].describe())


=== age ===


count    200000.000000
mean         40.173510
std          14.161414
min          16.000000
25%          28.000000
50%          40.000000
75%          52.000000
max          64.000000
Name: age, dtype: float64


=== duration_seconds ===


count    200000.000000
mean        721.858430
std         658.909189
min           0.000000
25%         229.000000
50%         541.000000
75%        1051.000000
max       10151.000000
Name: duration_seconds, dtype: float64


=== completion_rate ===


count    200000.000000
mean          0.811346
std           0.354383
min           0.000000
25%           0.840000
50%           1.000000
75%           1.000000
max           1.000000
Name: completion_rate, dtype: float64

... (4 outputs left out)


=== previous_online_courses ===


count    200000.000000
mean          4.981245
std           2.174642
min           0.000000
25%           3.000000
50%           5.000000
75%           6.000000
max          15.000000
Name: previous_online_courses, dtype: float64


=== instructor_rating ===


count    200000.000000
mean          4.238062
std           0.418330
min           3.500000
25%           3.900000
50%           4.200000
75%           4.600000
max           5.000000
Name: instructor_rating, dtype: float64

### 7.2 Validación de rangos esperados

Se aplican reglas básicas para verificar que determinadas variables se encuentren dentro de rangos razonables.

In [23]:
validaciones_rango = {
    "age": (0, 100),
    "duration_seconds": (0, None),
    "completion_rate": (0, 1),
    "score": (0, 1),
    "attempts": (0, None),
    "previous_online_courses": (0, None),
    "instructor_rating": (0, 5)
}

for columna, (minimo, maximo) in validaciones_rango.items():

    if columna not in df_integrado.columns:
        continue

    serie = df_integrado[columna].dropna()

    fuera_rango = pd.Series(False, index=serie.index)

    if minimo is not None:
        fuera_rango |= serie < minimo

    if maximo is not None:
        fuera_rango |= serie > maximo

    print(
        f"{columna:30} -> "
        f"Valores fuera de rango: {fuera_rango.sum()}"
    )

age                            -> Valores fuera de rango: 0
duration_seconds               -> Valores fuera de rango: 0
completion_rate                -> Valores fuera de rango: 0
score                          -> Valores fuera de rango: 0
attempts                       -> Valores fuera de rango: 0
previous_online_courses        -> Valores fuera de rango: 0
instructor_rating              -> Valores fuera de rango: 0


### 7.3 Validación de `score` y `completion_rate`

Las variables `score` y `completion_rate` utilizan una escala de 0 a 1.

Se valida que no existan valores negativos ni superiores a 1.

In [24]:
for columna in ["score", "completion_rate"]:

    serie = df_integrado[columna]

    print(f"\n=== {columna} ===")
    print("Mínimo:", serie.min())
    print("Máximo:", serie.max())

    fuera_rango = (
        serie.notna()
        &
        ~serie.between(0, 1)
    )

    print(
        "Registros fuera de rango:",
        fuera_rango.sum()
    )


=== score ===
Mínimo: 0.6
Máximo: 1.0
Registros fuera de rango: 0

=== completion_rate ===
Mínimo: 0.0
Máximo: 1.0
Registros fuera de rango: 0


### 7.4 Validación de edad

Se revisa la variable `age` para detectar valores imposibles o poco razonables dentro del contexto educativo.

In [25]:
print("Edad mínima:", df_integrado["age"].min())
print("Edad máxima:", df_integrado["age"].max())

print("\nDistribución:")
display(df_integrado["age"].describe())

Edad mínima: 16
Edad máxima: 64

Distribución:


count    200000.000000
mean         40.173510
std          14.161414
min          16.000000
25%          28.000000
50%          40.000000
75%          52.000000
max          64.000000
Name: age, dtype: float64

### 7.5 Validación de duración e intentos

Se verifica que las variables asociadas al comportamiento del estudiante no contengan valores negativos.

In [26]:
print(
    "Duraciones negativas:",
    (df_integrado["duration_seconds"] < 0).sum()
)

print(
    "Intentos negativos:",
    (df_integrado["attempts"] < 0).sum()
)

Duraciones negativas: 0
Intentos negativos: 0


In [27]:
display(
    df_integrado[
        ["duration_seconds", "attempts"]
    ].describe()
)

,duration_seconds,attempts
count,200000.000000,200000.000000
mean,721.858430,1.333140
std,658.909189,0.667106
min,0.000000,1.000000
25%,229.000000,1.000000
50%,541.000000,1.000000
75%,1051.000000,1.000000
max,10151.000000,3.000000


### 7.6 Validación de variables categóricas

Se revisan los valores presentes en las principales variables categóricas para detectar categorías inesperadas o inconsistencias de escritura.

In [28]:
variables_categoricas = [
    "activity_type",
    "device_used",
    "education_level",
    "study_field",
    "learning_style",
    "device_preference",
    "timezone",
    "motivation_level",
    "subject_area",
    "difficulty_level",
    "course_format"
]

for columna in variables_categoricas:

    if columna in df_integrado.columns:
        print(f"\n=== {columna} ===")
        display(
            df_integrado[columna]
            .value_counts(dropna=False)
            .to_frame("frecuencia")
        )


=== activity_type ===


,frecuencia
activity_type,
Note Taking,33754
Quiz Attempt,33411
Resource Download,33383
Discussion Post,33328
Assignment Submit,33170
Video Watch,32954



=== device_used ===


,frecuencia
device_used,
Mobile,99902
Desktop,80249
Tablet,19849



=== education_level ===


,frecuencia
education_level,
Undergraduate,79373
High School,59891
Graduate,40562
Professional,20174


... (12 outputs left out)


=== difficulty_level ===


,frecuencia
difficulty_level,
Beginner,84005
Intermediate,77458
Advanced,38537



=== course_format ===


,frecuencia
course_format,
Video,92172
Interactive,46340
Mixed,41440
Text-based,20048


### 7.7 Validación de espacios en variables de texto

Se verifica si existen espacios innecesarios al inicio o al final de los valores de texto.

In [29]:
columnas_texto = df_integrado.select_dtypes(
    include="object"
).columns

problemas_espacios = {}

for columna in columnas_texto:

    serie = df_integrado[columna].dropna().astype(str)

    cantidad = (
        serie != serie.str.strip()
    ).sum()

    if cantidad > 0:
        problemas_espacios[columna] = cantidad


if problemas_espacios:

    print("Columnas con espacios adicionales:")

    for columna, cantidad in problemas_espacios.items():
        print(f"{columna}: {cantidad}")

else:
    print("No se detectaron espacios adicionales.")

C:\Users\Rocket Consulting\AppData\Local\Temp\ipykernel_22748\967239872.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df_integrado.select_dtypes(


No se detectaron espacios adicionales.


### 7.8 Validación de variables binarias

Se revisan variables que representan condiciones de tipo verdadero/falso para confirmar que mantengan valores consistentes.

In [30]:
variables_booleanas = [
    "has_learning_disability",
    "has_assignments",
    "has_quizzes",
    "has_discussion_forum",
    "certificate_offered",
    "help_sought",
    "peer_interaction"
]

for columna in variables_booleanas:

    if columna in df_integrado.columns:
        print(f"\n=== {columna} ===")
        print(
            df_integrado[columna]
            .value_counts(dropna=False)
        )


=== has_learning_disability ===
has_learning_disability
0    170285
1     29715
Name: count, dtype: int64

=== has_assignments ===
has_assignments
1    153353
0     46647
Name: count, dtype: int64

=== has_quizzes ===
has_quizzes
1    182755
0     17245
Name: count, dtype: int64

=== has_discussion_forum ===
has_discussion_forum
1    116072
0     83928
Name: count, dtype: int64

=== certificate_offered ===
certificate_offered
1    135589
0     64411
Name: count, dtype: int64

=== help_sought ===
help_sought
False    169970
True      30030
Name: count, dtype: int64

=== peer_interaction ===
peer_interaction
False    189922
True      10078
Name: count, dtype: int64


### 7.9 Validación de fechas

Se verifica que la variable `timestamp` pueda interpretarse correctamente como fecha y hora.

In [31]:
timestamp_convertido = pd.to_datetime(
    df_integrado["timestamp"],
    errors="coerce"
)

fechas_invalidas = timestamp_convertido.isnull().sum()

print("Fechas inválidas:", fechas_invalidas)

print(
    "Fecha mínima:",
    timestamp_convertido.min()
)

print(
    "Fecha máxima:",
    timestamp_convertido.max()
)

Fechas inválidas: 0
Fecha mínima: 2024-06-12 13:55:11
Fecha máxima: 2025-06-12 13:50:19


### 7.10 Resumen de validaciones

In [32]:
resumen_validez = pd.DataFrame({
    "validacion": [
        "score fuera de rango",
        "completion_rate fuera de rango",
        "duration_seconds negativos",
        "attempts negativos",
        "timestamps inválidos"
    ],
    "registros_invalidos": [
        (
            ~df_integrado["score"]
            .dropna()
            .between(0, 1)
        ).sum(),

        (
            ~df_integrado["completion_rate"]
            .dropna()
            .between(0, 1)
        ).sum(),

        (
            df_integrado["duration_seconds"] < 0
        ).sum(),

        (
            df_integrado["attempts"] < 0
        ).sum(),

        fechas_invalidas
    ]
})

display(resumen_validez)

,validacion,registros_invalidos
0,score fuera de rango,0
1,completion_rate fuera de rango,0
2,duration_seconds negativos,0
3,attempts negativos,0
4,timestamps inválidos,0


### Interpretación del análisis de validez

Las variables numéricas analizadas se encuentran dentro de los rangos esperados y no se identificaron valores inválidos.

Las variables categóricas presentan dominios consistentes y no se observaron diferencias de escritura o categorías evidentemente incorrectas.

Las variables binarias utilizan representaciones de tipo `0/1` y `True/False`. Ambas son válidas, aunque podrán estandarizarse posteriormente durante la preparación de datos para análisis o modelado.

La variable `score` presenta valores entre 0.60 y 1.00. Este comportamiento deberá considerarse posteriormente al definir criterios de riesgo estudiantil, ya que no existen calificaciones inferiores a 0.60 dentro de los datos disponibles.

La variable `timestamp` pudo convertirse completamente a formato de fecha y comprende registros entre junio de 2024 y junio de 2025.

En general, no se identificaron problemas de validez que requieran corrección inmediata.

## 8. Detección y análisis de valores atípicos

En esta sección se analizan posibles valores atípicos en las principales variables numéricas.

La presencia de un valor atípico no implica necesariamente un error de calidad. Estos registros pueden representar comportamientos reales y relevantes de los estudiantes.

Por esta razón, los valores detectados serán analizados antes de considerar cualquier tratamiento o eliminación.

### 8.1 Distribución estadística de variables numéricas

Se revisan medidas descriptivas para identificar valores extremos y conocer la dispersión de las variables.

In [33]:
variables_outliers = [
    "age",
    "duration_seconds",
    "completion_rate",
    "score",
    "attempts",
    "previous_online_courses",
    "instructor_rating"
]

display(
    df_integrado[variables_outliers]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
age,200000.0,40.173510,14.161414,16.0,28.00,40.0,52.0,64.0
duration_seconds,200000.0,721.858430,658.909189,0.0,229.00,541.0,1051.0,10151.0
completion_rate,200000.0,0.811346,0.354383,0.0,0.84,1.0,1.0,1.0
score,66581.0,0.799858,0.115322,0.6,0.70,0.8,0.9,1.0
attempts,200000.0,1.333140,0.667106,1.0,1.00,1.0,1.0,3.0
previous_online_courses,200000.0,4.981245,2.174642,0.0,3.00,5.0,6.0,15.0
instructor_rating,200000.0,4.238062,0.418330,3.5,3.90,4.2,4.6,5.0


### 8.2 Detección de valores atípicos mediante IQR

Se utiliza el rango intercuartílico (IQR) para identificar observaciones alejadas de la distribución central.

Los límites se calculan mediante:

- límite inferior = Q1 - 1.5 × IQR
- límite superior = Q3 + 1.5 × IQR

In [34]:
def detectar_outliers_iqr(df, columna):

    serie = df[columna].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)

    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = serie[
        (serie < limite_inferior)
        | (serie > limite_superior)
    ]

    return {
        "variable": columna,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "outliers": len(outliers),
        "porcentaje_outliers": (
            len(outliers) / len(serie) * 100
        )
    }

In [35]:
resultados_outliers = []

for columna in variables_outliers:
    resultados_outliers.append(
        detectar_outliers_iqr(
            df_integrado,
            columna
        )
    )

df_outliers = pd.DataFrame(
    resultados_outliers
)

df_outliers[
    [
        "variable",
        "Q1",
        "Q3",
        "IQR",
        "limite_inferior",
        "limite_superior",
        "outliers",
        "porcentaje_outliers"
    ]
].round(2)

,variable,Q1,Q3,IQR,limite_inferior,limite_superior,outliers,porcentaje_outliers
0,age,28.00,52.0,24.00,-8.00,88.00,0,0.00
1,duration_seconds,229.00,1051.0,822.00,-1004.00,2284.00,5494,2.75
2,completion_rate,0.84,1.0,0.16,0.60,1.24,38396,19.20
3,score,0.70,0.9,0.20,0.40,1.20,0,0.00
4,attempts,1.00,1.0,0.00,1.00,1.00,44341,22.17
5,previous_online_courses,3.00,6.0,3.00,-1.50,10.50,2287,1.14
6,instructor_rating,3.90,4.6,0.70,2.85,5.65,0,0.00


### 8.3 Análisis de valores extremos

Se revisan los valores mínimos y máximos de las variables donde fueron identificados posibles outliers.

In [36]:
for columna in variables_outliers:

    resultado = detectar_outliers_iqr(
        df_integrado,
        columna
    )

    if resultado["outliers"] > 0:

        print(f"\n=== {columna} ===")

        print(
            "Límite inferior:",
            round(resultado["limite_inferior"], 2)
        )

        print(
            "Límite superior:",
            round(resultado["limite_superior"], 2)
        )

        print(
            "Cantidad de outliers:",
            resultado["outliers"]
        )

        print(
            "Mínimo real:",
            df_integrado[columna].min()
        )

        print(
            "Máximo real:",
            df_integrado[columna].max()
        )


=== duration_seconds ===
Límite inferior: -1004.0
Límite superior: 2284.0
Cantidad de outliers: 5494
Mínimo real: 0
Máximo real: 10151

=== completion_rate ===
Límite inferior: 0.6
Límite superior: 1.24
Cantidad de outliers: 38396
Mínimo real: 0.0
Máximo real: 1.0

=== attempts ===
Límite inferior: 1.0
Límite superior: 1.0
Cantidad de outliers: 44341
Mínimo real: 1
Máximo real: 3

=== previous_online_courses ===
Límite inferior: -1.5
Límite superior: 10.5
Cantidad de outliers: 2287
Mínimo real: 0
Máximo real: 15


### 8.4 Valores atípicos a nivel de estudiante

Se analizan los indicadores consolidados por estudiante para detectar comportamientos de participación o rendimiento significativamente diferentes al resto de la población.

In [37]:
variables_student_outliers = [
    "total_interactions",
    "total_duration_minutes",
    "average_completion_rate",
    "average_score",
    "evaluated_activities",
    "average_attempts",
    "help_sought_count",
    "peer_interaction_count"
]

resultados_student_outliers = []

for columna in variables_student_outliers:

    resultados_student_outliers.append(
        detectar_outliers_iqr(
            df_student_analysis,
            columna
        )
    )

df_student_outliers = pd.DataFrame(
    resultados_student_outliers
)

display(
    df_student_outliers[
        [
            "variable",
            "Q1",
            "Q3",
            "IQR",
            "limite_inferior",
            "limite_superior",
            "outliers",
            "porcentaje_outliers"
        ]
    ].round(2)
)

,variable,Q1,Q3,IQR,limite_inferior,limite_superior,outliers,porcentaje_outliers
0,total_interactions,22.00,28.00,6.00,13.00,37.00,96,1.20
1,total_duration_minutes,242.94,352.05,109.11,79.28,515.71,74,0.92
2,average_completion_rate,0.76,0.86,0.10,0.62,1.01,72,0.90
3,average_score,0.77,0.83,0.06,0.69,0.91,105,1.31
4,evaluated_activities,6.00,10.00,4.00,0.00,16.00,46,0.57
5,average_attempts,1.24,1.42,0.18,0.97,1.69,75,0.94
6,help_sought_count,2.00,5.00,3.00,-2.50,9.50,60,0.75
7,peer_interaction_count,0.00,2.00,2.00,-3.00,5.00,11,0.14


### Criterio de tratamiento de valores atípicos

Los valores identificados mediante IQR no serán eliminados automáticamente.

En un entorno de Learning Analytics, comportamientos extremos pueden representar casos reales de alto o bajo nivel de participación y podrían contener información relevante para identificar estudiantes en riesgo.

Por esta razón, los outliers serán conservados mientras no se identifique evidencia de que corresponden a errores de captura, procesamiento o integración.

### Interpretación de valores atípicos

El análisis mediante IQR identificó valores atípicos en varias variables, tanto a nivel de interacción como a nivel de estudiante.

En variables como `attempts`, el método IQR puede resultar especialmente sensible debido a la baja variabilidad de los datos. Por esta razón, los registros detectados no se consideran automáticamente errores.

En `completion_rate` se observa un porcentaje elevado de valores fuera de los límites definidos por IQR. Sin embargo, estos registros pueden representar actividades con bajo nivel de finalización y son potencialmente relevantes para el análisis de riesgo estudiantil.

A nivel de estudiante, los porcentajes de valores atípicos son bajos y se concentran en pequeños grupos con comportamientos diferentes al resto de la población.

Por lo tanto, no se eliminarán los outliers detectados. Estos registros se conservarán para etapas posteriores, ya que podrían representar perfiles de baja participación, bajo rendimiento o comportamiento atípico relevantes para la identificación de riesgo estudiantil.

## 9. Consistencia entre variables

En esta sección se validan relaciones lógicas entre diferentes variables del dataset.

El objetivo es detectar inconsistencias semánticas que podrían no identificarse mediante revisiones de valores nulos, rangos o duplicados.

### 9.1 Consistencia entre tipo de actividad y score

Se verifica que las actividades evaluables cuenten con una calificación y que las actividades no evaluables no presenten valores de `score`.

In [38]:
actividades_evaluables = [
    "Assignment Submit",
    "Quiz Attempt"
]

evaluables_sin_score = df_integrado[
    df_integrado["activity_type"].isin(actividades_evaluables)
    & df_integrado["score"].isna()
]

no_evaluables_con_score = df_integrado[
    ~df_integrado["activity_type"].isin(actividades_evaluables)
    & df_integrado["score"].notna()
]

print(
    "Actividades evaluables sin score:",
    len(evaluables_sin_score)
)

print(
    "Actividades no evaluables con score:",
    len(no_evaluables_con_score)
)

Actividades evaluables sin score: 0
Actividades no evaluables con score: 0


### 9.2 Consistencia entre quizzes y características del curso

Se valida que los registros de tipo `Quiz Attempt` correspondan a cursos configurados con quizzes.

In [39]:
quiz_inconsistentes = df_integrado[
    (df_integrado["activity_type"] == "Quiz Attempt")
    & (df_integrado["has_quizzes"] == 0)
]

print(
    "Quiz Attempt en cursos sin quizzes:",
    len(quiz_inconsistentes)
)

Quiz Attempt en cursos sin quizzes: 2898


### 9.3 Consistencia entre assignments y características del curso

Se valida que los registros de tipo `Assignment Submit` correspondan a cursos configurados con actividades evaluables de tipo assignment.

In [40]:
assignment_inconsistentes = df_integrado[
    (df_integrado["activity_type"] == "Assignment Submit")
    & (df_integrado["has_assignments"] == 0)
]

print(
    "Assignment Submit en cursos sin assignments:",
    len(assignment_inconsistentes)
)

Assignment Submit en cursos sin assignments: 7709


### 9.4 Consistencia entre foros y actividad de discusión

Se verifica que las publicaciones de discusión correspondan a cursos que cuenten con foro habilitado.

In [41]:
discussion_inconsistentes = df_integrado[
    (df_integrado["activity_type"] == "Discussion Post")
    & (df_integrado["has_discussion_forum"] == 0)
]

print(
    "Discussion Post en cursos sin foro:",
    len(discussion_inconsistentes)
)

Discussion Post en cursos sin foro: 14215


### 9.5 Consistencia entre dispositivo utilizado y preferencia

Se compara el dispositivo utilizado en cada interacción con la preferencia registrada del estudiante.

Una diferencia no representa necesariamente un error, ya que un estudiante puede utilizar distintos dispositivos.

In [42]:
df_integrado["device_matches_preference"] = (
    df_integrado["device_used"]
    == df_integrado["device_preference"]
)

coincidencia_dispositivo = (
    df_integrado["device_matches_preference"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(
    coincidencia_dispositivo
    .to_frame("porcentaje")
)

,porcentaje
device_matches_preference,
False,57.69
True,42.31


In [43]:
print(
    "Coincidencia promedio:",
    f"{df_integrado['device_matches_preference'].mean() * 100:.2f}%"
)

Coincidencia promedio: 42.31%


### 9.6 Consistencia de attempts según tipo de actividad

Se analiza la distribución de `attempts` por tipo de actividad para verificar si el comportamiento de esta variable es coherente con el contexto de cada interacción.

In [44]:
attempts_por_actividad = (
    df_integrado
    .groupby("activity_type")["attempts"]
    .agg(
        ["count", "min", "median", "mean", "max"]
    )
    .round(2)
)

display(attempts_por_actividad)

,count,min,median,mean,max
activity_type,,,,,
Assignment Submit,33170,1,2.0,2.0,3
Discussion Post,33328,1,1.0,1.0,1
Note Taking,33754,1,1.0,1.0,1
Quiz Attempt,33411,1,2.0,2.0,3
Resource Download,33383,1,1.0,1.0,1
Video Watch,32954,1,1.0,1.0,1


### 9.7 Completion rate por tipo de actividad

Se analiza el nivel de finalización según el tipo de actividad para detectar comportamientos inconsistentes o inesperados.

In [45]:
completion_por_actividad = (
    df_integrado
    .groupby("activity_type")["completion_rate"]
    .agg(
        ["count", "min", "mean", "median", "max"]
    )
    .round(4)
)

display(completion_por_actividad)

,count,min,mean,median,max
activity_type,,,,,
Assignment Submit,33170,0.0,0.8010,1.00,1.0
Discussion Post,33328,0.0,0.7949,1.00,1.0
Note Taking,33754,0.0,0.8021,1.00,1.0
Quiz Attempt,33411,1.0,1.0000,1.00,1.0
Resource Download,33383,0.0,0.8008,1.00,1.0
Video Watch,32954,0.0,0.6673,0.71,1.0


### 9.8 Resumen de consistencia

In [46]:
resumen_consistencia = pd.DataFrame({
    "validacion": [
        "Actividades evaluables sin score",
        "Actividades no evaluables con score",
        "Quiz Attempt en cursos sin quizzes",
        "Assignment Submit en cursos sin assignments",
        "Discussion Post en cursos sin foro"
    ],
    "registros_inconsistentes": [
        len(evaluables_sin_score),
        len(no_evaluables_con_score),
        len(quiz_inconsistentes),
        len(assignment_inconsistentes),
        len(discussion_inconsistentes)
    ]
})

display(resumen_consistencia)

,validacion,registros_inconsistentes
0,Actividades evaluables sin score,0
1,Actividades no evaluables con score,0
2,Quiz Attempt en cursos sin quizzes,2898
3,Assignment Submit en cursos sin assignments,7709
4,Discussion Post en cursos sin foro,14215


### Resultado del análisis de consistencia

Se evaluaron relaciones lógicas entre las variables de actividad, características del curso y comportamiento del estudiante.

Las inconsistencias detectadas no serán corregidas automáticamente. En caso de existir, deberán analizarse considerando el contexto del proceso educativo y la forma en que fueron generados los datos.

La comparación entre `device_used` y `device_preference` se utiliza únicamente como indicador descriptivo y no como regla de calidad estricta.

## 10. Interpretación de inconsistencias detectadas

Durante las validaciones de consistencia se identificaron algunos casos donde la actividad registrada no coincide con las características declaradas del curso:

- existen registros `Quiz Attempt` en cursos con `has_quizzes = 0`,
- existen registros `Assignment Submit` en cursos con `has_assignments = 0`,
- y existen registros `Discussion Post` en cursos con `has_discussion_forum = 0`.

Estas situaciones no serán corregidas automáticamente, ya que pueden deberse a diferencias entre la configuración del curso y el comportamiento registrado en la plataforma, o a inconsistencias propias de la fuente original.

También se observó que el dispositivo utilizado coincide con la preferencia del estudiante en aproximadamente 42.31% de las interacciones. Esta diferencia no se considera un problema de calidad, ya que la preferencia de dispositivo no implica que el estudiante utilice siempre el mismo medio de acceso.

Por lo tanto, estas observaciones se documentan como inconsistencias semánticas o diferencias de comportamiento, pero los registros se conservan para no perder información potencialmente relevante.

### 10.1 Resumen de hallazgos de calidad

In [47]:
resumen_final_calidad = pd.DataFrame({
    "dimension": [
        "Completitud",
        "Duplicados",
        "Unicidad",
        "Validez de rangos",
        "Fechas",
        "Valores categóricos",
        "Valores atípicos",
        "Consistencia semántica"
    ],
    "resultado": [
        "Adecuada; los nulos de score son esperados según el tipo de actividad",
        "No se detectaron duplicados completos relevantes",
        "Las llaves principales mantienen la unicidad esperada",
        "No se detectaron valores fuera de los rangos definidos",
        "No se detectaron timestamps inválidos",
        "Las categorías presentan valores consistentes",
        "Se detectaron outliers válidos que se conservarán",
        "Se detectaron algunas diferencias entre actividades y configuración del curso"
    ],
    "accion": [
        "Conservar los valores nulos válidos",
        "Sin acción",
        "Sin acción",
        "Sin acción",
        "Sin acción",
        "Sin acción",
        "Conservar y analizar posteriormente",
        "Documentar y conservar"
    ]
})

display(resumen_final_calidad)

,dimension,resultado,accion
0,Completitud,Adecuada; los nulos de score son esperados seg...,Conservar los valores nulos válidos
1,Duplicados,No se detectaron duplicados completos relevantes,Sin acción
2,Unicidad,Las llaves principales mantienen la unicidad e...,Sin acción
3,Validez de rangos,No se detectaron valores fuera de los rangos d...,Sin acción
4,Fechas,No se detectaron timestamps inválidos,Sin acción
5,Valores categóricos,Las categorías presentan valores consistentes,Sin acción
6,Valores atípicos,Se detectaron outliers válidos que se conservarán,Conservar y analizar posteriormente
7,Consistencia semántica,Se detectaron algunas diferencias entre activi...,Documentar y conservar


## 11. Conclusiones

El análisis de calidad permitió verificar que los datasets utilizados en el proyecto presentan una estructura consistente y adecuada para continuar con futuras etapas de análisis.

No se identificaron problemas graves de integridad, duplicidad, validez de rangos o formato de fechas.

Los valores nulos encontrados en `score` fueron identificados como valores válidos asociados a actividades que no generan calificación, por lo que no se realizó imputación.

Los valores atípicos detectados mediante el método IQR fueron conservados, ya que pueden representar comportamientos reales de los estudiantes y ser relevantes para análisis posteriores.

También se identificaron algunas inconsistencias semánticas entre determinadas actividades y las características declaradas del curso. Debido a que no existe evidencia suficiente para considerarlas errores, estos registros se conservaron y se documentaron como hallazgos de calidad.

En términos generales, los datos presentan un nivel adecuado de calidad para continuar con el proyecto EdTech Learning Analytics.

In [48]:
print("=== Validación final del proyecto ===")

print(
    "Dataset integrado:",
    "OK" if ruta_integrado.exists() else "NO ENCONTRADO"
)

print(
    "Dataset analítico:",
    "OK" if ruta_student_analysis.exists() else "NO ENCONTRADO"
)

print("\nNotebook de calidad finalizado correctamente.")

=== Validación final del proyecto ===
Dataset integrado: OK
Dataset analítico: OK

Notebook de calidad finalizado correctamente.
